In [2]:
import lightgbm as lgb

from lightgbm import LGBMClassifier

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix
)
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
import optuna

In [3]:
train_df = pd.read_csv("training(2003-2023).csv")
test_df = pd.read_csv("test(2024-25).csv")

print(train_df.shape)
print(test_df.shape)

(20988361, 14)
(1182188, 14)


In [4]:
FEATURES = [
    "CHLOR_A",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos",
    "LAT_scaled",
    "LON_scaled"
]

TARGET = "PHYTOBLOOM"

In [5]:
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

In [ ]:
lgb_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    random_state=42,
    n_estimators=200
)

lgb_model.fit(X_train, y_train)

In [9]:
train_pred = lgb_model.predict(X_train)
test_pred = lgb_model.predict(X_test)

NameError: name 'lgb_model' is not defined

In [9]:
print("TRAIN RESULTS\n")

print(classification_report(y_train, train_pred))


print("Macro F1:",
      f1_score(y_train, train_pred, average='macro'))


print("\n\nTEST RESULTS\n")

print(classification_report(y_test, test_pred))


print("Macro F1:",
      f1_score(y_test, test_pred, average='macro'))

TRAIN RESULTS

              precision    recall  f1-score   support

           0       1.00      0.99      1.00  16719377
           1       0.83      0.90      0.86   1069207
           2       0.94      0.93      0.94   3199777

    accuracy                           0.98  20988361
   macro avg       0.92      0.94      0.93  20988361
weighted avg       0.98      0.98      0.98  20988361

Macro F1: 0.9313158283817046


TEST RESULTS

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    956898
           1       0.79      0.89      0.84     75876
           2       0.93      0.88      0.90    149414

    accuracy                           0.98   1182188
   macro avg       0.91      0.92      0.91   1182188
weighted avg       0.98      0.98      0.98   1182188

Macro F1: 0.9132371684429893


In [10]:
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": lgb_model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance)

      Feature  Importance
1     day_sin        4022
0     CHLOR_A        3767
5  LAT_scaled        3583
6  LON_scaled        3513
2     day_cos        2801
3   month_sin         160
4   month_cos         154


In [12]:
cm = confusion_matrix(y_test, test_pred)

print(cm)

[[954528    212   2158]
 [    23  67789   8064]
 [  1261  17352 130801]]


In [19]:
def objective(trial):

    # params = {

    #     "n_estimators":
    #     trial.suggest_int(
    #         "n_estimators",
    #         40,
    #         350,
    #         step=50
    #     ),

    #     "max_depth":
    #     trial.suggest_int(
    #         "max_depth",
    #         2,
    #         8
    #     ),

    #     "learning_rate":
    #     trial.suggest_float(
    #         "learning_rate",
    #         0.03,
    #         0.15
    #     ),

    #     "min_child_weight":
    #     trial.suggest_int(
    #         "min_child_weight",
    #         10,
    #         18
    #     ),

    #     "subsample":
    #     trial.suggest_float(
    #         "subsample",
    #         0.50,
    #         0.85
    #     ),

    #     "colsample_bytree":
    #     trial.suggest_float(
    #         "colsample_bytree",
    #         0.65,
    #         0.80
    #     ),

    #     "gamma":
    #     trial.suggest_float(
    #         "gamma",
    #         0.0,
    #         0.05
    #     ),

    #     "lambda":
    #     trial.suggest_float(
    #         "lambda",
    #         0.001,
    #         0.01,
    #         log=True
    #     ),

    #     "alpha":
    #     trial.suggest_float(
    #         "alpha",
    #         0.001,
    #         0.02,
    #         log=True
    #     )
    # }

    params = {
    "objective": "multiclass",
    "num_class": 3,
    "random_state": 42,
    "verbosity": -1,

    "n_estimators": trial.suggest_int("n_estimators", 50, 300, step=50),

    "learning_rate": trial.suggest_float(
        "learning_rate", 0.03, 0.15
    ),

    "max_depth": trial.suggest_int("max_depth", 4, 12),

    "num_leaves": trial.suggest_int("num_leaves", 31, 255),

    "min_child_samples": trial.suggest_int(
        "min_child_samples", 10, 80
    ),

    "subsample": trial.suggest_float(
        "subsample", 0.6, 1.0
    ),

    "colsample_bytree": trial.suggest_float(
        "colsample_bytree", 0.6, 1.0
    ),

    "reg_alpha": trial.suggest_float(
        "reg_alpha", 1e-3, 1.0, log=True
    ),

    "reg_lambda": trial.suggest_float(
        "reg_lambda", 1e-3, 1.0, log=True
    ),

    "min_split_gain": trial.suggest_float(
        "min_split_gain", 0.0, 0.2
    )
    }   

    skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

    scores = []

    for train_idx, val_idx in skf.split(X_train,y_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = lgb.LGBMClassifier(
            **params
        )

        model.fit(
            X_tr,
            y_tr,
        )

        pred = model.predict(X_val)

        macro_f1 = f1_score(
            y_val,
            pred,
            average="macro"
        )

        scores.append(
            macro_f1
        )

    return np.mean(scores)

In [20]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

[I 2026-07-03 16:28:48,208] A new study created in memory with name: no-name-902afefb-0429-4015-a8c7-6150355be128


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-07-03 16:48:00,574] Trial 0 finished with value: 0.9378266514313587 and parameters: {'n_estimators': 300, 'learning_rate': 0.12461695185334923, 'max_depth': 10, 'num_leaves': 54, 'min_child_samples': 39, 'subsample': 0.7939044677824056, 'colsample_bytree': 0.7667881830479806, 'reg_alpha': 0.6801198645503895, 'reg_lambda': 0.0779495291327052, 'min_split_gain': 0.09457197678770174}. Best is trial 0 with value: 0.9378266514313587.
[I 2026-07-03 16:58:12,890] Trial 1 finished with value: 0.9346448084589193 and parameters: {'n_estimators': 200, 'learning_rate': 0.14660465575595308, 'max_depth': 4, 'num_leaves': 202, 'min_child_samples': 59, 'subsample': 0.7057861343982615, 'colsample_bytree': 0.9519665223259459, 'reg_alpha': 0.4190955203738543, 'reg_lambda': 0.2555727190614416, 'min_split_gain': 0.09833078805693048}. Best is trial 0 with value: 0.9378266514313587.
[I 2026-07-03 17:14:21,298] Trial 2 finished with value: 0.936689257831075 and parameters: {'n_estimators': 200, 'learni

In [21]:
study.best_params

{'n_estimators': 250,
 'learning_rate': 0.09964310975306116,
 'max_depth': 11,
 'num_leaves': 228,
 'min_child_samples': 27,
 'subsample': 0.9998318766440227,
 'colsample_bytree': 0.903730158781237,
 'reg_alpha': 0.00305068574840351,
 'reg_lambda': 0.4452293374642786,
 'min_split_gain': 0.16454073973737562}

In [13]:
import lightgbm as lgb

def objective_multi_lgbm(trial):
    params = {
        "objective": "multiclass",
        "num_class": 3,
        "metric": "multi_logloss",

        "n_estimators": trial.suggest_int("n_estimators", 50, 300, step=25),

        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.20, log=True),

        # Bias narrower than round 1 (max_depth=11) to test the shallow-tree hypothesis
        "max_depth": trial.suggest_int("max_depth", 3, 10),

        # num_leaves should stay well under 2^max_depth to control complexity;
        # searching it independently but capped lower than round 1's 228
        "num_leaves": trial.suggest_int("num_leaves", 15, 150),

        "min_child_samples": trial.suggest_int("min_child_samples", 10, 60),

        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),

        "min_split_gain": trial.suggest_float("min_split_gain", 1e-3, 1.0, log=True),

        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gaps, val_scores = [], []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr)

        train_f1 = f1_score(y_tr, model.predict(X_tr), average="macro")
        val_f1 = f1_score(y_val, model.predict(X_val), average="macro")

        val_scores.append(val_f1)
        gaps.append(train_f1 - val_f1)

    mean_val = np.mean(val_scores)
    mean_gap = np.mean(gaps)

    return mean_val, mean_gap

In [14]:
study_lgbm = optuna.create_study(directions=["maximize", "minimize"])
study_lgbm.optimize(objective_multi_lgbm, n_trials=50, show_progress_bar=True)

[I 2026-07-09 15:59:47,306] A new study created in memory with name: no-name-46615d1b-c58f-4dea-bc7a-ffb9ad770443


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-07-09 16:07:59,848] Trial 0 finished with values: [0.9335136230528327, 6.413648033034569e-05] and parameters: {'n_estimators': 75, 'learning_rate': 0.0650533904760636, 'max_depth': 8, 'num_leaves': 25, 'min_child_samples': 32, 'subsample': 0.6471532105936635, 'colsample_bytree': 0.9248642086788448, 'reg_alpha': 0.002064435808281518, 'reg_lambda': 0.0044610334692581355, 'min_split_gain': 0.26244146543189184}.
[I 2026-07-09 16:25:33,404] Trial 1 finished with values: [0.9374797773290859, 0.000518694630980443] and parameters: {'n_estimators': 250, 'learning_rate': 0.16405723670710584, 'max_depth': 9, 'num_leaves': 114, 'min_child_samples': 58, 'subsample': 0.7709004468158035, 'colsample_bytree': 0.9087520055787734, 'reg_alpha': 3.5073004737006888, 'reg_lambda': 0.15141988896882227, 'min_split_gain': 0.45901343212683604}.
[I 2026-07-09 16:42:37,594] Trial 2 finished with values: [0.9351108075760026, 0.00014305885079402003] and parameters: {'n_estimators': 175, 'learning_rate': 0.07

In [6]:
def objective_multi_lgbm_v3(trial):
    params = {
        "objective": "multiclass",
        "num_class": 3,
        "metric": "multi_logloss",

        # Tightened further — trial 3 (best gap+F1) used 75; keep search
        # centered there instead of allowing drift back toward 250
        "n_estimators": trial.suggest_int("n_estimators", 20, 100, step=5),

        # Nudged down to match the lower n_estimators ceiling
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.08, log=True),

        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "num_leaves": trial.suggest_int("num_leaves", 30, 140),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 65),

        "subsample": trial.suggest_float("subsample", 0.65, 0.90),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.75, 1.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 1.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 1e-3, 0.10, log=True),

        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gaps, val_scores = [], []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr)

        train_f1 = f1_score(y_tr, model.predict(X_tr), average="macro")
        val_f1 = f1_score(y_val, model.predict(X_val), average="macro")

        val_scores.append(val_f1)
        gaps.append(train_f1 - val_f1)

    return np.mean(val_scores), np.mean(gaps)

In [7]:
study_lgbm_v3 = optuna.create_study(directions=["maximize", "minimize"])
study_lgbm_v3.optimize(objective_multi_lgbm_v3, n_trials=25, show_progress_bar=True)

[I 2026-07-11 16:32:17,246] A new study created in memory with name: no-name-6e668a88-4512-4628-b5d5-f8a657fa2005


  0%|          | 0/25 [00:00<?, ?it/s]

[I 2026-07-11 16:41:19,009] Trial 0 finished with values: [0.9337973116920587, 6.787582836818019e-05] and parameters: {'n_estimators': 100, 'learning_rate': 0.053493379220834227, 'max_depth': 7, 'num_leaves': 32, 'min_child_samples': 27, 'subsample': 0.6756222400005168, 'colsample_bytree': 0.8149098047372301, 'reg_alpha': 0.01288252558120657, 'reg_lambda': 0.2650851447131036, 'min_split_gain': 0.003204377920104696}.
[I 2026-07-11 16:44:32,494] Trial 1 finished with values: [0.9097826547258574, 2.9641456892814943e-05] and parameters: {'n_estimators': 30, 'learning_rate': 0.023379087447394316, 'max_depth': 6, 'num_leaves': 39, 'min_child_samples': 53, 'subsample': 0.6553659806208614, 'colsample_bytree': 0.8890778610789003, 'reg_alpha': 0.006597739164788958, 'reg_lambda': 0.040782388830948896, 'min_split_gain': 0.04275593246555142}.
[I 2026-07-11 16:48:48,368] Trial 2 finished with values: [0.925066121842957, 0.00010877954851538086] and parameters: {'n_estimators': 45, 'learning_rate': 0.

In [11]:
best_trials = study_lgbm_v3.best_trials
i=0
for t in best_trials:
    print(f"{i} val_f1={t.values[0]:.4f}, gap={t.values[1]:.4f}, params={t.params}")
    i+=1

0 val_f1=0.9338, gap=0.0001, params={'n_estimators': 100, 'learning_rate': 0.053493379220834227, 'max_depth': 7, 'num_leaves': 32, 'min_child_samples': 27, 'subsample': 0.6756222400005168, 'colsample_bytree': 0.8149098047372301, 'reg_alpha': 0.01288252558120657, 'reg_lambda': 0.2650851447131036, 'min_split_gain': 0.003204377920104696}
1 val_f1=0.9307, gap=-0.0000, params={'n_estimators': 30, 'learning_rate': 0.05100951282436649, 'max_depth': 4, 'num_leaves': 66, 'min_child_samples': 34, 'subsample': 0.897944887767238, 'colsample_bytree': 0.7551178667909587, 'reg_alpha': 0.12087874480114016, 'reg_lambda': 0.11024868381878898, 'min_split_gain': 0.006075023874388366}
2 val_f1=0.9347, gap=0.0002, params={'n_estimators': 50, 'learning_rate': 0.07055314896878793, 'max_depth': 9, 'num_leaves': 113, 'min_child_samples': 39, 'subsample': 0.6824320114004905, 'colsample_bytree': 0.9461752355308758, 'reg_alpha': 0.011394527486291009, 'reg_lambda': 0.0539373828016813, 'min_split_gain': 0.0305290698

In [18]:
params=best_trials[1].params
params

{'n_estimators': 30,
 'learning_rate': 0.05100951282436649,
 'max_depth': 4,
 'num_leaves': 66,
 'min_child_samples': 34,
 'subsample': 0.897944887767238,
 'colsample_bytree': 0.7551178667909587,
 'reg_alpha': 0.12087874480114016,
 'reg_lambda': 0.11024868381878898,
 'min_split_gain': 0.006075023874388366}

In [16]:
for t in best_trials:
    params=t.params
    lgb_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    **params,
    random_state=42
    )

    lgb_model.fit(X_train, y_train)
    y_pred1 = lgb_model.predict(
        X_train
    )

    y_pred = lgb_model.predict(
        X_test
    )
    macro_f1_train = f1_score(
        y_train,
        y_pred1,
        average="macro"
    )

    macro_f1_test = f1_score(
        y_test,
        y_pred,
        average="macro"
    )
    gap=macro_f1_train-macro_f1_test
    print(f"{i} Gap: {gap} Train Macro F1 : {macro_f1_train}  Test Macro F1 : {macro_f1_test}")


7 Gap: 0.017193452065241033 Train Macro F1 : 0.9329927523191582  Test Macro F1 : 0.9157993002539172
7 Gap: 0.01201257199862571 Train Macro F1 : 0.9296942668815399  Test Macro F1 : 0.9176816948829142
7 Gap: 0.01879064854644308 Train Macro F1 : 0.9340398199589212  Test Macro F1 : 0.9152491714124781
7 Gap: 0.016475424134729022 Train Macro F1 : 0.9324232217601525  Test Macro F1 : 0.9159477976254234
7 Gap: 0.016633711419144936 Train Macro F1 : 0.9326542412382505  Test Macro F1 : 0.9160205298191055
7 Gap: 0.01702488309185146 Train Macro F1 : 0.9331235996653118  Test Macro F1 : 0.9160987165734603
7 Gap: 0.014210610112311284 Train Macro F1 : 0.9314067526823888  Test Macro F1 : 0.9171961425700775


In [19]:
lgb_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    **params,
    random_state=42
)

lgb_model.fit(X_train, y_train)



,num_leaves,66
,max_depth,4
,learning_rate,0.05100951282436649
,n_estimators,30
,objective,'multiclass'
,min_split_gain,0.006075023874388366
,min_child_samples,34
,subsample,0.897944887767238
,colsample_bytree,0.7551178667909587
,reg_alpha,0.12087874480114016
,reg_lambda,0.11024868381878898


In [22]:
import joblib

joblib.dump(lgb_model, r"D:\INCOIS\Notebooks\lgb_model_NAS.pkl")

['D:\\INCOIS\\Notebooks\\lgb_model_NAS.pkl']

In [20]:
y_pred1 = lgb_model.predict(
    X_train
)

y_pred = lgb_model.predict(
    X_test
)

In [21]:
print(
    classification_report(
        y_train,
        y_pred1
    )
)

macro_f1 = f1_score(
    y_train,
    y_pred1,
    average="macro"
)

print(
    "Train Macro F1:",
    macro_f1
)
print()
print(
    classification_report(
        y_test,
        y_pred
    )
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print(
    "Test Macro F1:",
    macro_f1
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00  16719377
           1       0.83      0.87      0.85   1069207
           2       0.94      0.94      0.94   3199777

    accuracy                           0.98  20988361
   macro avg       0.92      0.94      0.93  20988361
weighted avg       0.98      0.98      0.98  20988361

Train Macro F1: 0.9296942668815399

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    956898
           1       0.81      0.89      0.85     75876
           2       0.93      0.88      0.91    149414

    accuracy                           0.98   1182188
   macro avg       0.91      0.92      0.92   1182188
weighted avg       0.98      0.98      0.98   1182188

Test Macro F1: 0.9176816948829142


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    956898
           1       0.80      0.90      0.85     75876
           2       0.94      0.88      0.91    149414

    accuracy                           0.98   1182188
   macro avg       0.91      0.92      0.92   1182188
weighted avg       0.98      0.98      0.98   1182188

Test Macro F1: 0.916799488829394
